# Day 5 — Artificial Neural Network and final comparison

**Project:** RetailMind AI — Customer Churn Prediction

**Goal:** Prepare churn data for a feed-forward ANN, train and evaluate it, compare it with the classical champion, inspect learning curves and save project artifacts.



## 11. Day 5 — Keras MNIST warm-up demo

Load the MNIST handwritten-digit dataset and train a small neural network as a warm-up exercise. This demonstrates the basic TensorFlow/Keras workflow—loading arrays, normalizing image values, defining a network, compiling it and fitting it—before applying the same general training process to customer churn data.

In [7]:
import tensorflow as tf
tf.keras.utils.set_random_seed(RANDOM_STATE)

(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = tf.keras.datasets.mnist.load_data()
x_train_mnist = x_train_mnist.astype("float32") / 255.0
x_test_mnist = x_test_mnist.astype("float32") / 255.0

mnist_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])
mnist_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
mnist_model.fit(
    x_train_mnist, y_train_mnist,
    epochs=3, batch_size=128, validation_split=0.1, verbose=1
)
mnist_loss, mnist_accuracy = mnist_model.evaluate(x_test_mnist, y_test_mnist, verbose=0)
print(f"MNIST test accuracy: {mnist_accuracy:.4f}")


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8971 - loss: 0.3734 - val_accuracy: 0.9558 - val_loss: 0.1688
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9506 - loss: 0.1710 - val_accuracy: 0.9658 - val_loss: 0.1256
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9651 - loss: 0.1223 - val_accuracy: 0.9692 - val_loss: 0.1075
MNIST test accuracy: 0.9647


## 12. Day 5 — Prepare normalized churn inputs for the ANN

Transform the churn training and test features into numerical arrays suitable for a neural network. The preprocessor is fitted only on the training data, then reused to transform the test data. This prevents information from the test set from influencing preprocessing and ensures both sets have the same encoded feature layout.

In [ ]:
# Fit preprocessing on training rows only; transform test rows with the same fitted object.
ann_preprocessor = make_preprocessor(X_train)
X_train_ann = ann_preprocessor.fit_transform(X_train)
X_test_ann = ann_preprocessor.transform(X_test)

if hasattr(X_train_ann, "toarray"):
    X_train_ann = X_train_ann.toarray()
    X_test_ann = X_test_ann.toarray()

X_train_ann = np.asarray(X_train_ann, dtype="float32")
X_test_ann = np.asarray(X_test_ann, dtype="float32")

print("ANN training matrix:", X_train_ann.shape)
print("ANN test matrix:", X_test_ann.shape)


## 13. Day 5 — Build and train the feed-forward ANN

Define the feed-forward ANN architecture, configure its optimizer and loss function, and train it using the prepared churn features. The validation data helps monitor performance during training, while the network's hidden layers learn combinations of customer attributes that may be associated with churn.

In [ ]:
ann_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_ann.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)
ann_model.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

ann_history = ann_model.fit(
    X_train_ann, y_train,
    validation_split=0.20,
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)
print("Epochs completed:", len(ann_history.history["loss"]))


## 14. Day 5 — Evaluate ANN and compare with classical champion

Generate churn probabilities from the trained ANN, convert them into class predictions using the threshold in the code, and calculate the evaluation metrics. The results are placed alongside the classical champion so the ANN can be judged on the same test set and against the same business-relevant measures.

In [ ]:
ann_prob = ann_model.predict(X_test_ann, verbose=0).ravel()
ann_pred = (ann_prob >= 0.5).astype("int32")

ann_metrics = {
    "Model": "ANN",
    "Accuracy": accuracy_score(y_test, ann_pred),
    "Precision": precision_score(y_test, ann_pred, zero_division=0),
    "Recall": recall_score(y_test, ann_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, ann_prob)
}

classical_metrics = evaluate_classifier(
    classical_champion, X_test, y_test, classical_champion_name
)

day5_comparison = pd.DataFrame([classical_metrics, ann_metrics])
display(day5_comparison)
day5_comparison.to_csv(REPORTS_DIR / "day5_ann_comparison.csv", index=False)

print("ANN classification report:")
print(classification_report(y_test, ann_pred, zero_division=0))


## 15. Day 5 — Plot learning curves and save artifacts

Plot the ANN's training and validation history to inspect how its loss changes over epochs, then save the trained model and other project artifacts. The curves help identify whether training is still improving or whether the model may be overfitting. Saving the artifacts makes it possible to reuse the trained model without fitting it again.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ann_history.history["loss"], label="Training loss")
plt.plot(ann_history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("Day 5 ANN: Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / "day5_ann_loss.png", dpi=150)
plt.show()

ann_model.save(MODELS_DIR / "churn_ann_v5.keras")
joblib.dump(ann_preprocessor, MODELS_DIR / "churn_ann_preprocessor_v5.joblib")

print("Saved ANN:", MODELS_DIR / "churn_ann_v5.keras")
print("Saved ANN preprocessor:", MODELS_DIR / "churn_ann_preprocessor_v5.joblib")
print("Saved comparison:", REPORTS_DIR / "day5_ann_comparison.csv")


## Day 5 — Completion

This notebook contains the executable project workflow through Day 5. Run all cells in order and review the printed metrics, tables and plots before recording results in the project README.